Imports:

In [3]:
import os
import pandas as pd
import torch
import random
import numpy as np
import pytorch_lightning as pl
from sympy import false
from torch.utils.data import DataLoader
from tqdm  import tqdm
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader
torch.set_default_dtype(torch.float)
from utils.ClassicDistributedKalman import diffusion_extended_kalman_filter_parallel_edge
from utils.DistributedKalmanData import (HSystem, HSystemLinear, FSystem, FSystemLinear, GraphDataset, CreateGraph)
from utils.constant_vel_model import ConstantVelocityModel, DistanceAngleObservation,create_distance_based_graph

from utils.DistributedKalmanNet import GraphKalmanProcess, StateKnowledge
from utils.BaselineModels import GnnRnnLightning
from pytorch_lightning.loggers import CSVLogger

C:\Users\kfirg\PycharmProjects\distributedKNET2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def seed_everything(seed=42):
    """
    Set the random seed for reproducibility.
    :param seed: the random seed to set
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

Create System and Graph:

In [5]:
seed_everything(42)
num_nodes= 10#nodes in graph
state_dimension = 4
time_delta = 0.1#time interval between measurements
TIME_STEPS = 20#timesteps
node_positions = np.random.rand(num_nodes, 2) * 100#randomize node positions
R_SCALE = np.array([0.5 ])# node observation noise level
scale = 1
q = 1#state noise level
P0 = 1 * np.eye(state_dimension)#initial state covariance
x0 = scale * np.ones((state_dimension,1)).T#initial state
h_sys_linear =DistanceAngleObservation(node_positions)
f_sys_linear =ConstantVelocityModel(time_delta)
g = create_distance_based_graph(node_positions, 5)#create random  graph with node_num nodes and degree>=5


In [6]:
def plot_learning_curve(log_dir, r_value, save_dir="kfirmodels"):
    """
    Reads metrics.csv from the logger directory and plots
    Train vs Val loss curves for a given noise level r.
    """
    metrics_path = os.path.join(log_dir, "metrics.csv")
    df = pd.read_csv(metrics_path)

    plt.figure(figsize=(7,5))

    # Training loss
    if "train_loss" in df.columns:
        plt.plot(df["step"], df["train_loss"], label="Train Loss")

    # Validation loss
    if "val_loss" in df.columns:
        val_df = df.dropna(subset=["val_loss"])
        plt.plot(val_df["step"], val_df["val_loss"], label="Val Loss")

    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"Learning Curve (r = {r_value})")
    plt.legend()
    plt.grid(True)

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"learning_curve_r={r_value}.png")
    plt.savefig(save_path, dpi=200)
    plt.close()

    print(f"Saved learning curve: {save_path}")

In [ ]:
seed_everything(42)
BATCH_SIZE = 64
repo_path=r"C:\Users\kfirg\PycharmProjects\distributedKNET2"#change to your path
save_dir =os.path.join(repo_path,"models/kfiritay")
for r in R_SCALE:
    logger = CSVLogger("kfirlogs", name=f"kalman_no_consensus_r={r}",version="0_kfir")#log
# training process
    #create dataset for each noise level:
    r_array=r*np.ones(num_nodes)
    train_dataset_linear = GraphDataset(g, f_sys_linear, h_sys_linear, q, r_array, monte_carlo_simulations=10000, time_steps=TIME_STEPS, n_expansions=1, x0=scale,state_dim=state_dimension)
    val_dataset_linear = GraphDataset(g , f_sys_linear, h_sys_linear, q, r_array, monte_carlo_simulations=256, time_steps=TIME_STEPS, n_expansions=1, x0=scale,state_dim=state_dimension)
    kalman_process=GraphKalmanProcess(f_sys_linear,signal_dim=state_dimension,edge_features_dim=1,node_kalman_dim=state_dimension**2,edge_kalman_dim=2,hidden_dim=64,lr=1e-5,r_array=r,learn_edge_kalman=False,x0_scale=scale)#creat nn
    kalman_process=kalman_process.to(torch.float)
    train_loader = DataLoader(train_dataset_linear, shuffle=True, batch_size=BATCH_SIZE)#create data in training ready shape
    val_loader=DataLoader(val_dataset_linear,shuffle=False,batch_size=BATCH_SIZE)
#define early stopping  criteria for validation loss
    early_stopping = pl.callbacks.EarlyStopping(
    monitor='val_loss:_epoch', patience=5, verbose=True, mode='min', min_delta=0.01)#stop when reaching difference in validation loss of min_delta
    trainer = pl.Trainer(max_epochs=25, accelerator='auto',logger=logger, log_every_n_steps=5,callbacks=[early_stopping], gradient_clip_val=1)
    trainer.fit(kalman_process, train_loader, val_loader)
    log_dir=logger.log_dir
    torch.save(kalman_process.state_dict(),f'kfirmodels/no_consensus/kalman_process_linear_no_mismatch_r={r}.pth')#save model values
    plot_learning_curve(log_dir, r)#plot learning curve for each noise level

